# Oppitunti 09 - Metakognition suunnittelumalli


## Käyttöönotto

Tämä muistikirja havainnollistaa metakognition suunnittelumallia käyttäen Microsoft Agent Frameworkia.

**Esivaatimukset:**
- Azure OpenAI -käyttöönotto konfiguroituna ympäristömuuttujien kautta
- Azure CLI todennettu (`az login`)


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Mikä on metakognitio?

Metakognitio on **ajattelun ajattelemista**. Tekoälyagenttien yhteydessä se tarkoittaa agenttien rakentamista, jotka pystyvät:

- **Itsetarkastella** omia tuloksiaan ja päättelyprosessiaan
- **Tunnistaa virheitä** ja toipua hallitusti sen sijaan, että epäonnistuisi huomaamatta
- **Arvioida** ovatko niiden vastaukset täydellisiä ja hyödyllisiä
- **Mukauttaa** strategiaansa, kun alkuperäinen lähestymistapa ei toimi (esim. siirtymällä varajärjestelmään)

Metakognitiivinen agentti ei vain vastaa kysymyksiin — se seuraa omaa suorituskykyään ja mukautuu lennossa.


## Ensisijaiset ja varatyökalut

Yleinen metakognition malli on **varautumisstrategia**. Agentti yrittää ensin ensisijaista työkalua; jos se epäonnistuu (esim. 404-virhe), agentti tunnistaa virheen ja vaihtaa läpinäkyvästi varatyökaluun.

Tämä heijastaa todellisen maailman järjestelmiä, joissa ensisijaiset palvelut saattavat olla poissa käytöstä ja agenttien on itse diagnosoitava ongelma ennen vaihtoehtoisen polun valitsemista.

Alla määrittelemme kaksi lentojen hakutyökalua:
- **Primary** — kattaa Pariisin, Tokion ja Barcelonan
- **Backup** — kattaa Berliinin, Sydneyn ja New Yorkin


In [ ]:
@tool(approval_mode="never_require")
def get_flight_times(
    destination: Annotated[str, "The destination city"]
) -> str:
    """Get available flight times for a destination (primary source)."""
    flights = {
        "Paris": "Departures: 08:00, 12:30, 17:45 — from $350",
        "Tokyo": "Departures: 11:00, 23:30 — from $890",
        "Barcelona": "Departures: 07:15, 14:00, 19:30 — from $280",
    }
    if destination in flights:
        return flights[destination]
    raise Exception(f"404: No flights found for {destination} in primary system")


@tool(approval_mode="never_require")
def get_flight_times_backup(
    destination: Annotated[str, "The destination city"]
) -> str:
    """Get available flight times from backup system (used when primary fails)."""
    backup_flights = {
        "Berlin": "Departures: 09:00, 16:00 — from $220",
        "Sydney": "Departures: 22:00 — from $1200",
        "New York City": "Departures: 06:00, 10:30, 15:00, 20:00 — from $450",
    }
    return backup_flights.get(
        destination,
        f"No flights found for {destination} in any system. Please try again later.",
    )

## Itsetarkastava agentti virheenkorjauksella

Alla olevaa agenttia on ohjeistettu kokeilemaan ensin ensisijaista lentojärjestelmää, tunnistamaan virheet ja siirtymään läpinäkyvästi varajärjestelmään. Jokaisen vastauksen jälkeen se arvioi lyhyesti itse, vastasiko se täysin käyttäjän kysymykseen.


In [ ]:
agent = await provider.create_agent(
    tools=[get_flight_times, get_flight_times_backup],
    name="FlightBookingAgent",
    instructions="""You are a flight booking agent with self-reflection capabilities.

When looking up flights:
1. Try the primary flight system first (get_flight_times)
2. If the primary system fails (404 error), acknowledge the error and try the backup system (get_flight_times_backup)
3. Always explain to the user what happened — be transparent about fallbacks
4. If both systems fail, apologize and suggest alternatives

After each response, briefly evaluate whether your answer was complete and helpful.""",
)

# Test with a destination in primary system
print("=== Test 1: Destination in primary system ===")
response = await agent.run(
    "What flights are available to Paris?",
    )
print(response)

# Test with a destination only in backup system
print("\n=== Test 2: Destination only in backup system ===")
response = await agent.run(
    "What flights are available to Berlin?",
    )
print(response)

## Itsearvioinnin malli

Metakognition toinen puoli on **itsearviointi**: erillinen agentti (tai sama agentti toisella läpikäynnillä) arvioi vastausta täydellisyyden, tarkkuuden ja hyödyllisyyden suhteen.

Alla luomme `ResponseEvaluator` agentin, joka pisteyttää matka-agentin vastaukset kolmella osa-alueella.


In [ ]:
evaluation_agent = await provider.create_agent(
    tools=[get_flight_times, get_flight_times_backup],
    name="ResponseEvaluator",
    instructions="""You are a quality evaluator for travel agent responses.
Given a travel question and the agent's response, evaluate:
1. Completeness: Did it answer all parts of the question? (1-5)
2. Accuracy: Is the information correct? (1-5)
3. Helpfulness: Would a traveler find this useful? (1-5)
Provide a brief evaluation with scores and one suggestion for improvement.""",
)

# Evaluate the agent's response from Test 1
eval_prompt = f"""Question: What flights are available to Paris?
Agent Response: {response}

Please evaluate the above response."""

evaluation = await evaluation_agent.run(eval_prompt)
print("=== Self-Evaluation ===")
print(evaluation)

## Yhteenveto

Tässä oppitunnissa opit rakentamaan **metakognitiivisia agenteja** Microsoft Agent Frameworkin avulla:

- **Itsetutkiskelu**: Agentit, jotka seuraavat omaa päättelyään ja viestivät läpinäkyvästi, mitä tapahtui.
- **Virheiden toipuminen vararatkaisuilla**: Pää- ja varatyökalumalli, jossa agentti havaitsee epäonnistumiset (esim. 404-virheet) ja yrittää automaattisesti vaihtoehtoista lähdettä.
- **Itsearviointi**: Erillinen arvioija-agentti, joka pisteyttää vastaukset täydellisyyden, tarkkuuden ja hyödyllisyyden mukaan.

Nämä mallit tekevät agenteista kestävämpiä, läpinäkyvämpiä ja luotettavampia — tärkeitä ominaisuuksia tuotantoon siirrettäessä.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
Vastuuvapauslauseke:
Tämä asiakirja on käännetty käyttämällä tekoälypohjaista käännöspalvelua Co-op Translator (https://github.com/Azure/co-op-translator). Vaikka pyrimme tarkkuuteen, huomioithan, että automaattiset käännökset voivat sisältää virheitä tai epätarkkuuksia. Alkuperäistä asiakirjaa sen alkuperäisellä kielellä tulee pitää auktoritatiivisena lähteenä. Tärkeän tiedon osalta suositellaan ammattimaista ihmiskäännöstä. Emme ole vastuussa mahdollisista väärinkäsityksistä tai virheellisistä tulkinnoista, jotka johtuvat tämän käännöksen käytöstä.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
